# JackSparrow v43 multi-head — Delta Exchange India (Colab)

Train one **multi-head** bundle (`metadata_v43.json` + `model_artifact_v43.pkl`) with intraday horizons:

| Key | Forward bars | ~Minutes |
|-----|----------------|----------|
| scalp_10m | 2 | 10m |
| intraday_30m | 6 | 30m |
| trend_1h | 12 | 1h |
| swing_2h | 24 | 2h |

Local export: `python scripts/train_v43_multihead_export.py --feature-csv <path>`


## References (in your cloned repo)

- `docs/03-ml-models.md` — model storage and v43 path
- `docs/feature_entrypoints_audit.md` — train/serve entrypoints
- `feature_store/jacksparrow_v43_contract.py` — feature order + version gate
- `feature_store/jacksparrow_v43_build_matrix.py` — `build_v43_feature_matrix`
- `agent/models/v43_pickle_shims.py` — `EnsembleModel`, `LGBMModel`, `FeatureEngineer` pickle contract


In [21]:
# Colab deps for agent + feature_store imports (align with agent/requirements.txt)
%pip install -q \
    "numpy>=1.24" "pandas>=2.0" "httpx>=0.27" "structlog==23.2.0" \
    "pydantic>=2.5" "pydantic-settings>=2.1" "python-dotenv>=1.0" \
    "joblib>=1.3" "scikit-learn>=1.3" "xgboost==2.0.2" "lightgbm==4.1.0" \
    requests


## 1) Repository on `sys.path`

**Preferred:** just run the next cell — it clones the **`major-rework`** branch automatically and switches to it if an older clone is already present.

Manual equivalent:

```bash
git clone --branch major-rework https://github.com/energyforreal/JackSparrow.git /content/trading-agent
```

If you omit the target directory, Git creates `/content/JackSparrow` — the next cell detects that too.

(Private repo: use a [personal access token](https://github.com/settings/tokens) or SSH URL instead.)

This notebook needs **`feature_store/jacksparrow_v43_build_matrix.py`** and **`jacksparrow_v43_contract.py`** on the branch you clone. If the repo-root cell fails after `git pull`, push those files from your local checkout.

Alternatively mount Google Drive containing a checkout and set environment variable `TRADING_AGENT_ROOT`
to that folder before importing.


In [ ]:
import subprocess
from pathlib import Path

_CLONE = Path("/content/trading-agent")
_URL = "https://github.com/energyforreal/JackSparrow.git"
_BRANCH = "major-rework"

if (_CLONE / ".git").is_dir():
    print("Repo already present:", _CLONE)
    result = subprocess.run(
        ["git", "-C", str(_CLONE), "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True, text=True,
    )
    current_branch = result.stdout.strip()
    if current_branch != _BRANCH:
        print(f"Switching from '{current_branch}' to '{_BRANCH}'...")
        subprocess.run(["git", "-C", str(_CLONE), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(_CLONE), "checkout", _BRANCH], check=True)
        subprocess.run(["git", "-C", str(_CLONE), "pull", "origin", _BRANCH], check=True)
    else:
        subprocess.run(["git", "-C", str(_CLONE), "pull", "origin", _BRANCH], check=True)
    print(f"On branch '{_BRANCH}', up to date.")
else:
    subprocess.run(["git", "clone", "--branch", _BRANCH, _URL, str(_CLONE)], check=True)
    print(f"Cloned branch '{_BRANCH}' to", _CLONE)


## 1b) If the repo-root cell fails: wrong branch cloned

Diagnostics showing **`/content/trading-agent/.../jacksparrow_v43_build_matrix.py` exists: False** mean Colab cloned the **wrong branch** — the v43 modules only exist on `major-rework`, not `main`.

**Fix A — re-run the §1 clone cell (handles this automatically).**

The §1 clone cell now always clones / switches to `major-rework`. Simply re-run it, then re-run this repo-root detection cell.

If you need to fix it manually in Colab (new code cell):

```python
import subprocess
subprocess.run(["git", "-C", "/content/trading-agent", "fetch", "origin"], check=True)
subprocess.run(["git", "-C", "/content/trading-agent", "checkout", "major-rework"], check=True)
subprocess.run(["git", "-C", "/content/trading-agent", "pull", "origin", "major-rework"], check=True)
print("Done — now re-run from the repo-root detection cell")
```

**Fix B (no-GitHub upload):** run the **next** code cell once, upload the required `.py` files from your PC’s `feature_store/` folder, then **re-run** the repo-root detection cell.


In [ ]:
# Colab only: upload v43 modules if they are not on GitHub yet.
# Set RUN_V43_UPLOAD = True, run this cell, pick the two files from your PC, then re-run the repo-root cell.

RUN_V43_UPLOAD = False

from pathlib import Path

if not RUN_V43_UPLOAD:
    print("Skipping upload (set RUN_V43_UPLOAD = True to upload from PC).")
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload path only works on Google Colab. Use git push on your PC.") from exc

    dest = Path("/content/trading-agent/feature_store")
    dest.mkdir(parents=True, exist_ok=True)
    print("Upload: jacksparrow_v43_build_matrix.py and jacksparrow_v43_contract.py")
    for name, data in files.upload().items():
        base = Path(name).name
        if not base.lower().endswith(".py"):
            continue
        out = dest / base
        out.write_bytes(data)
        print("Wrote", out, "(", out.stat().st_size, "bytes)")
    print("Done. Re-run the repo-root cell above, then continue.")


In [ ]:
# Optional helper cell left intentionally blank.
# (No-op)

In [ ]:
import os
import sys
from pathlib import Path

# Colab: uncomment if your clone is not auto-detected
# os.environ["TRADING_AGENT_ROOT"] = "/content/trading-agent"

_MARKER = Path("feature_store") / "jacksparrow_v43_build_matrix.py"

def _repo_has_marker(c: Path) -> bool:
    try:
        return (c / _MARKER).is_file()
    except OSError:
        return False


candidates: list[Path] = []
if os.environ.get("TRADING_AGENT_ROOT"):
    candidates.append(Path(os.environ["TRADING_AGENT_ROOT"]).resolve())
# Default clone targets (see section 1). `git clone <url>` alone creates /content/<repo_name>/.
for fixed in ("/content/trading-agent", "/content/JackSparrow", "/content/jacksparrow"):
    candidates.append(Path(fixed).resolve())
candidates.append(Path.cwd().resolve())
p = Path.cwd().resolve()
for _ in range(16):
    candidates.append(p)
    if p == p.parent:
        break
    p = p.parent
_content = Path("/content")
if _content.is_dir():
    try:
        _skip_names = frozenset({"sample_data", ".config"})
        for child in sorted(_content.iterdir()):
            if child.is_dir() and not child.name.startswith(".") and child.name not in _skip_names:
                candidates.append(child.resolve())
    except OSError:
        pass

seen: set[Path] = set()
deduped: list[Path] = []
for c in candidates:
    try:
        r = c.resolve()
    except OSError:
        continue
    if r not in seen:
        seen.add(r)
        deduped.append(r)

REPO_ROOT = None
for c in deduped:
    if _repo_has_marker(c):
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    listing = ""
    if _content.is_dir():
        try:
            names = sorted(
                p.name for p in _content.iterdir() if p.is_dir() and not p.name.startswith(".")
            )
            listing = f" Top-level folders under /content: {names!r}."
        except OSError:
            pass
    print(f"Diagnosing: Checking for {_MARKER} in detected candidates.")
    for c in deduped:
        expected_path = c / _MARKER
        print(f"  Checking: {expected_path} (exists: {expected_path.exists()}, is_file: {expected_path.is_file()})")
    raise FileNotFoundError(
        f"Could not find repo root (need {_MARKER.as_posix()}). "
        "Push that file to GitHub (or merge branch that has it), then `git pull` under your clone, "
        "or set TRADING_AGENT_ROOT to a checkout that contains feature_store/. See notebook section 1b (git push or Colab upload)."
        + listing
    )

sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
# Colab safety: force-sync to latest major-rework and verify required symbols exist
import subprocess
from pathlib import Path

REPO = Path("/content/trading-agent")
BRANCH = "major-rework"

if not (REPO / ".git").is_dir():
    raise RuntimeError(f"Repo missing at {REPO}. Run the clone cell first.")

subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"], check=True)

sha = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("Checked out SHA:", sha)

required_checks = {
    "feature_store/jacksparrow_v43_oi_history.py": [
        "oi_feature_matrix_diagnostics",
        "align_ticker_frame_to_primary",
    ],
    "feature_store/jacksparrow_v43_train_multihead.py": [
        "resolve_min_dynamic_threshold",
        "V43_MIN_THRESHOLD_COST_FRACTION",
    ],
    "feature_store/jacksparrow_v43_build_matrix.py": [
        "align_ticker_frame_to_primary",
    ],
}

for rel, tokens in required_checks.items():
    p = REPO / rel
    if not p.is_file():
        raise FileNotFoundError(f"Missing required file: {p}")
    txt = p.read_text(encoding="utf-8")
    for tok in tokens:
        if tok not in txt:
            raise RuntimeError(f"Outdated file: {rel} (missing token: {tok})")

print("Repo is up-to-date and required implementations are present.")

In [ ]:
# Optional: verify v43 training modules exist on REPO_ROOT before heavy downloads
from pathlib import Path

_fs = Path(REPO_ROOT) / "feature_store"
for name in (
    "jacksparrow_v43_build_matrix.py",
    "jacksparrow_v43_contract.py",
    "jacksparrow_v43_train_multihead.py",
    "jacksparrow_v43_multihead.py",
):
    p = _fs / name
    print(name, "OK" if p.is_file() else "MISSING — push from your PC then git pull in Colab")


In [ ]:
import importlib.metadata as importlib_metadata
import json
import os
import platform
import subprocess
import time
import warnings
from datetime import datetime, timezone

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor

from feature_store.jacksparrow_v43_build_matrix import build_v43_feature_matrix
from feature_store.jacksparrow_v43_contract import (
    V43_CANONICAL_FEATURES,
    V43_COMPATIBLE_FEATURE_VERSION,
    validate_v43_metadata_compatibility,
)
from feature_store.jacksparrow_v43_multihead import (
    V43_MULTIHEAD_ARTIFACT_FORMAT,
    V43_MULTIHEAD_MODEL_FAMILY,
)
from feature_store.jacksparrow_v43_train_multihead import (
    artifact_dict_from_bundle,
    train_multihead_from_feature_matrix,
)
from agent.models.v43_pickle_shims import (
    EnsembleModel,
    FeatureEngineer,
    LGBMModel,
    MultiHeadBundle,
    set_v43_build_feature_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)



## 2) Delta Exchange India — historical candles

Public `GET /v2/history/candles` on `https://api.india.delta.exchange` (same as
`agent/data/historical_data_loader.py`). Use `page_size` ≤ 2000 and polite delays between calls.

Default pull: **200,000** five-minute candles (~694 days) and matching funding lookback.
Large pulls can take **15–30 minutes** on Colab depending on rate limits.


In [ ]:
DELTA_BASE = os.environ.get("DELTA_EXCHANGE_BASE_URL", "https://api.india.delta.exchange")
SYMBOL = os.environ.get("DELTA_SYMBOL", "BTCUSD")
RESOLUTION_5M = "5m"

TARGET_CANDLES_5M = int(os.environ.get("TARGET_CANDLES_5M", "200000"))
FUNDING_LOOKBACK_HOURS = int(os.environ.get("FUNDING_LOOKBACK_HOURS", "16000"))

REQUEST_DELAY_S = float(os.environ.get("DELTA_REQUEST_DELAY_S", "0.35"))
PAGE_SIZE = 2000

RESOLUTION_SECONDS = {"5m": 300, "15m": 900, "30m": 1800, "1h": 3600, "2h": 7200}


def _ts_col(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in ("open", "high", "low", "close", "volume"):
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["timestamp"] = pd.to_datetime(out["time"], unit="s", utc=True)
    out = out.drop(columns=["time"], errors="ignore")
    return out.sort_values("timestamp").reset_index(drop=True)


def fetch_candles_range(session, symbol, resolution, start_ts, end_ts, max_retries=4):
    url = f"{DELTA_BASE}/v2/history/candles"
    params = {
        "symbol": symbol,
        "resolution": resolution,
        "start": int(start_ts),
        "end": int(end_ts),
        "page_size": PAGE_SIZE,
    }
    last_exc: Exception = RuntimeError("no attempts made")
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            if not data.get("success"):
                raise RuntimeError(data)
            rows = data.get("result") or []
            return pd.DataFrame(rows)
        except (requests.exceptions.RequestException, RuntimeError) as exc:
            last_exc = exc
            if attempt < max_retries - 1:
                backoff = 2 ** attempt
                print(f"  fetch_candles_range retry {attempt + 1}/{max_retries - 1} in {backoff}s: {exc}")
                time.sleep(backoff)
    raise last_exc


def fetch_5m_history(symbol, target_rows, resolution="5m"):
    session = requests.Session()
    session.headers.update({"Accept": "application/json"})
    sec = RESOLUTION_SECONDS[resolution]
    end_ts = int(time.time())
    span = int(target_rows * sec * 1.15)
    start_ts = end_ts - span
    chunk_sec = PAGE_SIZE * sec
    est_chunks = max(1, int((end_ts - start_ts) / chunk_sec))
    print(f"  target_rows={target_rows:,} (~{target_rows * sec / 86400:.0f} days), est. API chunks: {est_chunks}")
    frames = []
    cursor = start_ts
    chunk_i = 0
    while cursor < end_ts:
        chunk_i += 1
        chunk_end = min(end_ts, cursor + chunk_sec)
        if chunk_i == 1 or chunk_i % 25 == 0 or chunk_end >= end_ts:
            print(f"  chunk {chunk_i}/{est_chunks} ...")
        df = fetch_candles_range(session, symbol, resolution, cursor, chunk_end)
        if not df.empty:
            frames.append(df)
        cursor = chunk_end
        time.sleep(REQUEST_DELAY_S)
    if not frames:
        return pd.DataFrame()
    raw = pd.concat(frames, ignore_index=True)
    raw = raw.drop_duplicates(subset=["time"]).sort_values("time")
    if len(raw) > target_rows:
        raw = raw.iloc[-target_rows:]
    return _ts_col(raw)


def fetch_funding_hourly(symbol, lookback_hours):
    session = requests.Session()
    session.headers.update({"Accept": "application/json"})
    end_ts = int(time.time())
    start_ts = end_ts - int(lookback_hours * 3600)
    frames = []
    cursor = start_ts
    while cursor < end_ts:
        chunk_end = min(end_ts, cursor + PAGE_SIZE * 3600)
        df = fetch_candles_range(session, f"FUNDING:{symbol}", "1h", cursor, chunk_end)
        if not df.empty:
            frames.append(df)
        cursor = chunk_end
        time.sleep(REQUEST_DELAY_S)
    if not frames:
        return pd.DataFrame(columns=["timestamp", "funding_rate"])
    raw = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["time"]).sort_values("time")
    raw["timestamp"] = pd.to_datetime(raw["time"], unit="s", utc=True)
    raw["funding_rate"] = pd.to_numeric(raw["close"], errors="coerce")
    return raw[["timestamp", "funding_rate"]].reset_index(drop=True)


print("Fetching 5m OHLCV ...")
df_5m = fetch_5m_history(SYMBOL, TARGET_CANDLES_5M, RESOLUTION_5M)
print("5m rows:", len(df_5m), "range:", df_5m["timestamp"].min(), "->", df_5m["timestamp"].max())

print(f"Fetching MARK:{SYMBOL} 5m candles for basis features ...")
df_mark = fetch_5m_history(f"MARK:{SYMBOL}", TARGET_CANDLES_5M, RESOLUTION_5M)
print("MARK rows:", len(df_mark), "range:", df_mark["timestamp"].min() if len(df_mark) else None, "->", df_mark["timestamp"].max() if len(df_mark) else None)

print("Fetching funding (1h) ...")
try:
    df_funding = fetch_funding_hourly(SYMBOL, FUNDING_LOOKBACK_HOURS)
    print("funding rows:", len(df_funding))
except Exception as _fund_exc:
    print(f"WARNING: funding fetch failed ({_fund_exc!r}). Continuing with empty funding data.")
    df_funding = pd.DataFrame(columns=["timestamp", "funding_rate"])

# Historical open interest (Delta OI:SYMBOL 5m candles — same API as OHLCV)
FETCH_HISTORICAL_OI = os.environ.get("V43_FETCH_HISTORICAL_OI", "true").lower() in (
    "1", "true", "yes",
)
df_oi_hist = pd.DataFrame()
if FETCH_HISTORICAL_OI:
    print(f"Fetching OI:{SYMBOL} 5m history for oi_contracts ...")
    try:
        _oi_raw = fetch_5m_history(f"OI:{SYMBOL}", TARGET_CANDLES_5M, RESOLUTION_5M)
        if len(_oi_raw):
            from feature_store.jacksparrow_v43_oi_history import oi_candles_to_ticker_frame

            df_oi_hist = oi_candles_to_ticker_frame(
                _oi_raw,
                df_mark=df_mark,
                df_spot=df_5m,
                align_to=df_5m,
            )
            print(
                "OI history rows:", len(df_oi_hist),
                "oi_contracts max:", float(df_oi_hist["oi_contracts"].max()),
            )
        else:
            print("WARNING: OI:SYMBOL candle fetch returned no rows.")
    except Exception as _oi_exc:
        print(f"WARNING: historical OI fetch failed ({_oi_exc!r}).")
else:
    print("Skipping historical OI fetch (V43_FETCH_HISTORICAL_OI=false).")



In [ ]:
# Optional: upload a locally fetched OI/microstructure CSV from scripts/fetch_oi_microstructure.py
# This is recommended for real bid/ask columns (`best_bid`, `best_ask`, `bid_size`, `ask_size`).
#
# from google.colab import files
# uploaded = files.upload()  # select oi_microstructure_BTCUSD.csv
# os.environ["V43_OI_HISTORY_CSV"] = "/content/oi_microstructure_BTCUSD.csv"
# os.environ["V43_ALLOW_EMPTY_OI_FOR_TRAINING"] = "false"
# print("Configured V43_OI_HISTORY_CSV and strict OI requirement")

## 2b) Derivatives-state inputs (OI + microstructure + basis)

JackSparrow v3 uses four information classes:

| Class | Source | Training in this notebook |
|-------|--------|---------------------------|
| OHLCV + funding | `df_5m`, `df_funding` | Fetched below |
| Open interest | `OI:BTCUSD` 5m candles + CSV / ring buffer | Fetched in §2 (`df_oi_hist`) or `V43_OI_HISTORY_CSV` |
| Basis | `MARK:BTCUSD` 5m candles + spot | `df_mark` fetched below |
| Microstructure | Bid/ask, spread from ticker | Zero-filled unless `V43_TICKER_SNAPSHOTS_CSV` or live ticker |

**Default (§2):** historical OI contracts via Delta `OI:{symbol}` 5m candles (same paginated API as OHLCV).
Delta ticker API returns **current** snapshot only for bid/ask. For full microstructure history, supply a CSV with at least
`timestamp`, `oi_contracts`, `oi_value_usd` and optional ticker columns (`mark_price`, `spot_price`, `bid_size`, `ask_size`, `best_bid`, `best_ask`, `price_band_upper`, `price_band_lower`, `predicted_funding_rate`).

- Export from live agent: `from agent.core.v43_oi_frames import export_oi_ring_buffer`
- Set `V43_OI_HISTORY_CSV` or `V43_TICKER_SNAPSHOTS_CSV` to your collected file path (best for production bundles)
- The next cell enforces `V43_ALLOW_EMPTY_OI_FOR_TRAINING=false` (strict mode): training fails if real OI/ticker history cannot be loaded.
- Provide `V43_OI_HISTORY_CSV` (recommended) or ensure `df_oi_hist` from `OI:{symbol}` fetch is available before training.
- Export live ring buffer (repo): `python scripts/export_v43_oi_ring_buffer.py --symbol BTCUSD --out oi_history.csv`

In [ ]:
import asyncio
import time
from pathlib import Path

from agent.core.v43_oi_frames import TICKER_RING_COLUMNS, fetch_oi_snapshot

# Strict mode: never proceed with zero-filled OI/microstructure features.
os.environ.setdefault("V43_ALLOW_EMPTY_OI_FOR_TRAINING", "false")
# Do not set V43_THRESHOLD_PERCENTILE (legacy p75); use explicit long/short percentiles:
os.environ.pop("V43_THRESHOLD_PERCENTILE", None)
os.environ.setdefault("V43_USE_META_STACK", "false")
os.environ.setdefault("V43_THRESHOLD_PERCENTILE_LONG", "90")
os.environ.setdefault("V43_THRESHOLD_PERCENTILE_SHORT", "10")
# Use percentile-only thresholds during evaluation (no cost-floor clamp).
os.environ.setdefault("V43_MIN_THRESHOLD_COST_FRACTION", "0.0")
# Optional: relax scalp gate once corr is consistently positive in research reruns.
# os.environ["V43_MIN_VALIDATION_CORR_SCALP_10M"] = "0.03"

OI_HISTORY_CSV = os.environ.get(
    "V43_OI_HISTORY_CSV",
    os.environ.get("V43_TICKER_SNAPSHOTS_CSV", ""),
).strip()
df_oi = pd.DataFrame(columns=list(TICKER_RING_COLUMNS))


def _normalize_oi_df(raw: pd.DataFrame) -> pd.DataFrame:
    out = raw.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True)
    for col in (
        "oi_contracts",
        "oi_value_usd",
        "mark_price",
        "spot_price",
        "best_bid",
        "best_ask",
        "bid_size",
        "ask_size",
        "price_band_upper",
        "price_band_lower",
        "predicted_funding_rate",
    ):
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if "taker_buy_ratio" not in out.columns:
        out["taker_buy_ratio"] = 0.5
    return out


if OI_HISTORY_CSV and Path(OI_HISTORY_CSV).is_file():
    df_oi = _normalize_oi_df(pd.read_csv(OI_HISTORY_CSV))
    print(f"Loaded ticker/OI history: {len(df_oi)} rows from {OI_HISTORY_CSV}")
elif "df_oi_hist" in dir() and isinstance(df_oi_hist, pd.DataFrame) and not df_oi_hist.empty:
    df_oi = _normalize_oi_df(df_oi_hist)
    print(
        f"Using historical OI:{SYMBOL} 5m candles: {len(df_oi)} rows "
        f"(oi_contracts max={float(df_oi['oi_contracts'].max()):.0f})"
    )
else:
    _fetch_live = os.environ.get("V43_FETCH_LIVE_TICKER_IF_NO_CSV", "true").lower() in (
        "1",
        "true",
        "yes",
    )
    _allow_empty = os.environ.get("V43_ALLOW_EMPTY_OI_FOR_TRAINING", "").lower() in (
        "1",
        "true",
        "yes",
    )
    live_df = pd.DataFrame(columns=list(TICKER_RING_COLUMNS))
    if _fetch_live:
        try:
            snap = asyncio.run(fetch_oi_snapshot(None, SYMBOL))
            has_data = (
                float(snap.get("oi_contracts") or 0) > 0
                or float(snap.get("oi_value_usd") or 0) > 0
                or float(snap.get("mark_price") or 0) > 0
            )
            if has_data:
                bar_s = 300
                ts = (int(time.time()) // bar_s) * bar_s
                live_df = _normalize_oi_df(pd.DataFrame([{**snap, "timestamp": ts}]))
        except Exception as exc:
            print(f"WARNING: live ticker fetch failed ({exc!r}).")

    if not live_df.empty:
        df_oi = live_df
        print(
            "Using live Delta ticker snapshot (sparse OI history). "
            "For production, export ring buffer and set V43_OI_HISTORY_CSV."
        )
    elif _allow_empty:
        print(
            "WARNING: training without ticker history — OI/microstructure features will be zero-filled."
        )
    else:
        raise RuntimeError(
            "Real ticker/OI history is required for training (strict mode). Options:\n"
            "  - Set V43_OI_HISTORY_CSV (timestamp + oi_contracts and optional bid/ask columns)\n"
            "  - Keep V43_FETCH_LIVE_TICKER_IF_NO_CSV=true and ensure Delta public ticker returns data\n"
            "  - Re-run §2 historical OI fetch so df_oi_hist is non-empty"
        )

## 3) Features and multi-head labels (v43 contract v5)

- Features: `build_v43_feature_matrix(..., df_oi=df_oi, df_mark=df_mark)` then `V43_CANONICAL_FEATURES` (53 features, v5).
- Contract version: `jacksparrow_v43_features_v5` (`V43_COMPATIBLE_FEATURE_VERSION`, 53 features).
- Training knobs (env, optional): `V43_THRESHOLD_PERCENTILE_LONG=90`, `V43_THRESHOLD_PERCENTILE_SHORT=10`, `V43_MIN_THRESHOLD_COST_FRACTION=0.0` (percentile-only thresholds in notebook eval), `V43_USE_META_STACK=false` (default **regressor_mean**), `cost_aware_labels=false` (gross forward returns — fees in agent gate 5), `V43_FETCH_HISTORICAL_OI=true`, `V43_MAKER_FEE` / `V43_SLIPPAGE`.
- Set `V43_USE_META_STACK=true` only with real OI/ticker history (`V43_OI_HISTORY_CSV`); otherwise meta-calibrator thresholds often collapse below round-trip cost.
- Labels: per-head **gross** forward returns via `train_multihead_from_feature_matrix` (2/6/12/24 bars); agent gate 5 filters economics at runtime.
- Export: single `MultiHeadBundle` artifact + `horizons{}` metadata (see §5).


In [ ]:
df_feat = build_v43_feature_matrix(
    df_5m,
    None,
    None,
    df_funding,
    df_oi=df_oi,
    df_mark=df_mark,
    for_training=True,
    primary_interval="5m",
)

missing = [f for f in V43_CANONICAL_FEATURES if f not in df_feat.columns]
if missing:
    raise RuntimeError(f"Feature matrix missing v3 columns: {missing}")
print("Feature contract:", V43_COMPATIBLE_FEATURE_VERSION, "n=", len(V43_CANONICAL_FEATURES))
print(
    f"Feature matrix: {df_feat.shape[0]} rows x {len(V43_CANONICAL_FEATURES)} features "
    f"({V43_COMPATIBLE_FEATURE_VERSION})"
)

from feature_store.jacksparrow_v43_oi_history import oi_feature_matrix_diagnostics

_oi_diag = oi_feature_matrix_diagnostics(df_feat)
print("OI feature std (0 => alignment failure):", _oi_diag)
if max(_oi_diag.values()) < 1e-8:
    print(
        "WARNING: OI features have no variance — check df_oi alignment "
        "(re-run §2 OI fetch + this cell)."
    )

feat_cols = list(V43_CANONICAL_FEATURES)
if "timestamp" in df_feat.columns:
    _label_idx = pd.to_datetime(df_feat["timestamp"], utc=True)
else:
    _label_idx = pd.to_datetime(df_5m["timestamp"], utc=True).iloc[: len(df_feat)]
_close_5m = pd.Series(
    pd.to_numeric(df_5m["close"], errors="coerce").values,
    index=pd.to_datetime(df_5m["timestamp"], utc=True),
)
close = _close_5m.reindex(_label_idx)

_maker_fee = float(os.environ.get("V43_MAKER_FEE", "0.0002"))  # Delta India BTC perp maker = 2 bps
_slippage = float(os.environ.get("V43_SLIPPAGE", "0.0003"))
_leverage = int(os.environ.get("V43_LEVERAGE", "3"))
_train_rng = int(os.environ.get("V43_TRAIN_RNG", "42"))

# Regressor-mean stack only (LGBM + XGB + RF); no meta-classifier or Ridge calibrator.
_use_meta = os.environ.get("V43_USE_META_STACK", "false").strip().lower() in (
    "1",
    "true",
    "yes",
)
_stack_label = "meta_calibrator" if _use_meta else "regressor_mean"
print(f"V43_USE_META_STACK={_use_meta} (default false — export stack: {_stack_label})")

multi_bundle, metadata = train_multihead_from_feature_matrix(
    df_feat[feat_cols],
    close,
    feat_cols=feat_cols,
    validation_fraction=float(os.environ.get("V43_VALIDATION_FRACTION", "0.15")),
    rng=_train_rng,
    maker_fee=_maker_fee,
    slippage=_slippage,
    leverage=_leverage,
    cost_aware_labels=False,
    use_meta_stack=_use_meta,
)

validation_metrics = metadata["horizons"]["scalp_10m"]["validation_metrics"]
split_metadata = metadata.get("split", {})
primary_ensemble = multi_bundle.get_head(2)  # scalp_10m primary
print("Multi-head training complete. Horizons:", list(metadata.get("horizons", {}).keys()))
print("Primary execution head (scalp_10m / 2 bars) validation_corr:", validation_metrics.get("validation_corr"))
print("round_trip_cost_pct:", metadata.get("runtime_cost_assumptions", {}).get("round_trip_cost_pct"))


In [ ]:
from feature_store.jacksparrow_v43_multihead import V43_HORIZON_KEYS, V43_HORIZON_KEY_TO_BARS
from feature_store.jacksparrow_v43_train_multihead import (
    V43_HORIZON_COST_SCALE,
    build_cost_aware_forward_labels,
)

print("=" * 84)
print("Post-train horizon diagnostics (prediction distribution + label balance)")
print("=" * 84)

for _hkey in V43_HORIZON_KEYS:
    _hb = metadata["horizons"][_hkey]
    _vm = _hb.get("validation_metrics", {})
    _fb = int(_hb.get("forward_bars") or V43_HORIZON_KEY_TO_BARS[_hkey])

    _split = _hb.get("split", {})
    _val_frac = float(_split.get("validation_fraction", 0.15))

    _cost_scale = float(V43_HORIZON_COST_SCALE.get(_fb, 1.0))
    _eff_cost = _round_trip = float(metadata.get("runtime_cost_assumptions", {}).get("round_trip_cost_pct") or 0.0) * _cost_scale

    _y_raw, _stats = build_cost_aware_forward_labels(close, _fb, round_trip_cost=_eff_cost)
    _work = df_feat[list(feat_cols)].copy()
    _work["target"] = _y_raw.values
    _work = _work.replace([np.inf, -np.inf], np.nan).dropna()

    _n_long = int((_work["target"] > 0).sum())
    _n_short = int((_work["target"] < 0).sum())
    _ls_ratio = float(_n_long / max(1, _n_short))

    _head = multi_bundle.get_head(_fb)
    _val_rows = int(_split.get("rows_validation") or 0)
    _x_val = _work.iloc[-_val_rows:][list(feat_cols)] if _val_rows > 0 else _work.iloc[0:0][list(feat_cols)]

    _inf = _vm.get("inference_path", "regressor_mean")
    if len(_x_val) and _inf == "regressor_mean":
        _pred = np.asarray(
            _head.predict(_x_val.values, X_df=_x_val, inference_stack="regressor_mean"),
            dtype=np.float64,
        ).ravel()
    elif len(_x_val):
        _pred = np.asarray(_head.predict(_x_val.values, X_df=_x_val), dtype=np.float64).ravel()
    else:
        _pred = np.array([], dtype=np.float64)

    if _pred.size:
        _p10 = float(np.nanpercentile(_pred, 10))
        _p90 = float(np.nanpercentile(_pred, 90))
        _mu = float(np.nanmean(_pred))
        _sd = float(np.nanstd(_pred))
    else:
        _p10 = _p90 = _mu = _sd = float("nan")

    print(
        f"[{_hkey}] pred_mean={_mu:.6f} pred_std={_sd:.6f} "
        f"p10={_p10:.6f} p90={_p90:.6f} "
        f"dyn_thr={float(_vm.get('dynamic_threshold') or 0.0):.6f} "
        f"short_thr={float(_vm.get('short_threshold') or 0.0):.6f} "
        f"tradable_frac={float(_vm.get('tradable_label_fraction') or 0.0):.4f} "
        f"labels_long={_n_long} labels_short={_n_short} long_short_ratio={_ls_ratio:.2f}"
    )


In [ ]:
from feature_store.jacksparrow_v43_multihead import format_horizon_training_diagnostics

print(format_horizon_training_diagnostics(metadata))


## 4) Legacy single-head cells (skipped)

Per-head training and walk-forward analysis run inside
`feature_store/jacksparrow_v43_train_multihead.py` for all four horizons in one export.
The Colab path uses **regressor mean** only (`use_meta_stack=False`); no meta-classifier stack.


In [ ]:
# Legacy single-head training removed — see multi-head train cell above.


## 4c) Inference stack (regressor mean — default)

By default (`V43_USE_META_STACK=false`), each horizon head is the unweighted mean of
LGBM + XGB + RF base regressors (`inference_path=regressor_mean`). Predictions stay in
forward-return scale so validation thresholds align with round-trip cost.

**Optional meta stack:** set `os.environ["V43_USE_META_STACK"] = "true"` before the §3
training cell only when you have real OI/ticker history (`V43_OI_HISTORY_CSV`). Without it,
the Ridge calibrator often compresses signals below cost. Match runtime with
`JACKSPARROW_V43_INFERENCE_STACK=regressor_mean` for default exports.


In [ ]:
# Meta-classifier stacking removed from the default Colab path (see §4c).
# Training runs in the §3 cell with use_meta_stack=False unless V43_USE_META_STACK=true.


In [ ]:
# Legacy single-head training removed — see multi-head train cell above.


In [ ]:
# Layer A (ML) vs Layer B (policy / gate-5) — separate market learning from execution economics
from feature_store.jacksparrow_v43_multihead import V43_HORIZON_KEYS, V43_HORIZON_KEY_TO_BARS
from feature_store.jacksparrow_v43_train_multihead import (
    build_forward_labels,
    chronological_split_indices,
)
from agent.core.v43_signal_gates import (
    gate5_long_edge_metrics,
    gate5_short_edge_metrics,
    round_trip_cost_pct,
)

_gate5_ratio = float(os.environ.get("JACKSPARROW_V43_MIN_EDGE_COST_RATIO", "0.2"))
_rtc = round_trip_cost_pct()

print("=" * 84)
print("Layer A — ML quality (predict market; gross forward return)")
print("=" * 84)
_layer_a = []
for _hkey in V43_HORIZON_KEYS:
    _vm = metadata["horizons"][_hkey]["validation_metrics"]
    _layer_a.append(
        {
            "horizon": _hkey,
            "corr_train_target": _vm.get("validation_corr"),
            "corr_gross": _vm.get("validation_corr_gross"),
            "dir_acc": _vm.get("directional_accuracy"),
            "tradable_frac": _vm.get("tradable_label_fraction"),
        }
    )
print(pd.DataFrame(_layer_a).to_string(index=False))

print("\n" + "=" * 84)
print("Layer B — Gate-5 pass rate on validation candidates (execution economics)")
print("=" * 84)
_layer_b = []
for _hkey in V43_HORIZON_KEYS:
    _vm = metadata["horizons"][_hkey]["validation_metrics"]
    _lc = _vm.get("long_candidates") or {}
    _sc = _vm.get("short_candidates") or {}
    _n_long = int(_lc.get("count") or 0)
    _n_short = int(_sc.get("count") or 0)
    _dt = float(_vm.get("dynamic_threshold") or 0.0)
    _st = float(_vm.get("short_threshold") or 0.0)
    _fb = int(metadata["horizons"][_hkey].get("forward_bars") or V43_HORIZON_KEY_TO_BARS[_hkey])
    _head = multi_bundle.get_head(_fb)
    _split = metadata["horizons"][_hkey]["split"]
    _val_frac = float(_split.get("validation_fraction", 0.15))
    _wk = df_feat[list(feat_cols)].copy()
    _wk["target_gross"] = build_forward_labels(close, _fb).values
    _wk = _wk.replace([np.inf, -np.inf], np.nan).dropna(subset=["target_gross"])
    if len(_wk) < 500:
        _layer_b.append({"horizon": _hkey, "gate5_long_pass": None, "gate5_short_pass": None})
        continue
    _, _, _vi, _ = chronological_split_indices(
        len(_wk), validation_fraction=_val_frac, embargo_bars=_fb
    )
    _Xv = _wk.iloc[_vi][list(feat_cols)]
    _pred = np.asarray(
        _head.predict(_Xv.values, X_df=_Xv, inference_stack="regressor_mean"),
        dtype=np.float64,
    ).ravel()
    _long_mask = _pred > _dt
    _short_mask = _pred < -abs(_st)
    _g5_long = sum(
        gate5_long_edge_metrics(float(p), _dt).passes for p in _pred[_long_mask]
    )
    _g5_short = sum(
        gate5_short_edge_metrics(float(p), _st).passes for p in _pred[_short_mask]
    )
    _layer_b.append(
        {
            "horizon": _hkey,
            "raw_long_cand": int(_long_mask.sum()),
            "raw_short_cand": int(_short_mask.sum()),
            "gate5_long_pass": int(_g5_long),
            "gate5_short_pass": int(_g5_short),
            "gate5_long_rate": float(_g5_long / max(1, _long_mask.sum())),
            "gate5_short_rate": float(_g5_short / max(1, _short_mask.sum())),
            "rtc": _rtc,
            "min_edge_ratio": _gate5_ratio,
        }
    )
print(pd.DataFrame(_layer_b).to_string(index=False))
print(
    "\nLayer B uses agent gate-5 (edge vs round_trip_cost). "
    "P&L below should use per-entry fees only — not per-bar round-trip stacking."
)


## 4d) Model Performance & Efficiency Summary

Pre-export evaluation of the trained multi-head bundle:

1. **Regression + trade-quality table** — all four horizons (`validation_corr`, RMSE/MAE, hit rates, tradable label fraction).
2. **Simulated validation P&L** — primary head (`scalp_10m`, 2 bars) with long/short thresholds from training metadata.
3. **Charts** — correlation by horizon, cumulative P&L, hit-rate comparison, prediction distribution (requires `matplotlib`; text-only fallback otherwise).

Cell 16 (`format_horizon_training_diagnostics`) remains a compact post-train sanity check; this section is the deeper gate before §5 export.

In [ ]:
from feature_store.jacksparrow_v43_multihead import V43_HORIZON_KEYS, V43_HORIZON_KEY_TO_BARS
from feature_store.jacksparrow_v43_train_multihead import (
    V43_HORIZON_COST_SCALE,
    build_cost_aware_forward_labels,
    build_forward_labels,
    chronological_split_indices,
    compute_v43_round_trip_cost_pct,
)

_runtime = metadata.get("runtime_cost_assumptions", {})
_round_trip = float(_runtime.get("round_trip_cost_pct") or 0.0)
_cost_aware = metadata.get("target_definition") == "cost_aware_forward_return"
_val_frac = float(
    metadata["horizons"]["scalp_10m"]["split"].get("validation_fraction", 0.15)
)


def _fmt_pct(x, scale=100.0):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "n/a"
    return f"{float(x) * scale:.2f}%"


def _fmt_f(x, digits=4):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "n/a"
    return f"{float(x):.{digits}f}"


# --- 1) Per-horizon regression + trade-quality table ---
_perf_rows = []
for _hkey in V43_HORIZON_KEYS:
    _hb = metadata.get("horizons", {}).get(_hkey, {})
    _vm = _hb.get("validation_metrics", {})
    _sp = _hb.get("split", {})
    _lc = _vm.get("long_candidates") or {}
    _sc = _vm.get("short_candidates") or {}
    _meta_auc = _vm.get("meta_auc")
    _perf_rows.append(
        {
            "horizon": _hkey,
            "bars": int(_hb.get("forward_bars") or V43_HORIZON_KEY_TO_BARS.get(_hkey, 0)),
            "val_rows": int(_sp.get("rows_validation") or 0),
            "corr": _fmt_f(_vm.get("validation_corr")),
            "RMSE": _fmt_f(_vm.get("validation_rmse"), 6),
            "MAE": _fmt_f(_vm.get("validation_mae"), 6),
            "dir_acc": _fmt_pct(_vm.get("directional_accuracy")),
            "meta_auc": _fmt_f(_meta_auc) if _meta_auc is not None else "n/a",
            "tradable%": _fmt_pct(_vm.get("tradable_label_fraction")),
            "L_hit_net": _fmt_pct(_lc.get("hit_rate_net_positive")),
            "S_hit_net": _fmt_pct(_sc.get("hit_rate_net_positive")),
            "L_net_ret%": _fmt_pct(_lc.get("net_mean_return")),
            "S_net_ret%": _fmt_pct(_sc.get("net_mean_return")),
        }
    )

perf_df = pd.DataFrame(_perf_rows)
print("=" * 72)
print("Multi-head validation metrics (chronological holdout per horizon)")
print("=" * 72)
print(perf_df.to_string(index=False))
print(
    f"\nCost assumptions: round_trip={_round_trip:.4%} | "
    f"target={metadata.get('target_definition')} | "
    f"inference={validation_metrics.get('inference_path')}"
)

# Profit-factor style summary per horizon (validation, active trades only)
_pf_rows = []
for _hkey in V43_HORIZON_KEYS:
    _hb = metadata.get("horizons", {}).get(_hkey, {})
    _vm = _hb.get("validation_metrics", {})
    _lc = _vm.get("long_candidates") or {}
    _sc = _vm.get("short_candidates") or {}
    _n_long = int(_lc.get("n_candidates") or 0)
    _n_short = int(_sc.get("n_candidates") or 0)
    _avg_win = float(_lc.get("net_mean_return") or 0.0)
    _avg_loss = float(_sc.get("net_mean_return") or 0.0)
    _pf = float("nan")
    if _avg_loss < 0.0:
        _pf = _avg_win / abs(_avg_loss) if _avg_win > 0.0 else 0.0
    _pf_rows.append(
        {
            "horizon": _hkey,
            "n_long": _n_long,
            "n_short": _n_short,
            "avg_long_net%": _fmt_pct(_avg_win),
            "avg_short_net%": _fmt_pct(_avg_loss),
            "profit_factor": _fmt_f(_pf),
        }
    )
pf_df = pd.DataFrame(_pf_rows)
print("\nProfit-factor summary (validation, active trades only)")
print(pf_df.to_string(index=False))

# --- 2) Simulated P&L on primary head (conditional pivot when scalp under-performs) ---
_scalp_vm = metadata["horizons"]["scalp_10m"]["validation_metrics"]
_scalp_fb = int(metadata["horizons"]["scalp_10m"].get("forward_bars", 2))
_scalp_coverage = float((_scalp_vm.get("long_candidates") or {}).get("coverage") or 0.0)
_scalp_sharpe_proxy = float(_scalp_vm.get("validation_corr") or 0.0)

# If scalp remains weak, evaluate notebook P&L on intraday_30m for comparison.
if _scalp_sharpe_proxy < 0.0 or _scalp_coverage < 0.03:
    _PRIMARY_KEY = "intraday_30m"
    _PRIMARY_FB = int(metadata["horizons"][_PRIMARY_KEY].get("forward_bars", 6))
else:
    _PRIMARY_KEY = "scalp_10m"
    _PRIMARY_FB = _scalp_fb

_primary_vm = metadata["horizons"][_PRIMARY_KEY]["validation_metrics"]
_primary_split = metadata["horizons"][_PRIMARY_KEY]["split"]
_long_thr = float(_primary_vm["dynamic_threshold"])
_short_thr = float(_primary_vm["short_threshold"])

_cost_scale = float(V43_HORIZON_COST_SCALE.get(_PRIMARY_FB, 1.0))
_effective_cost = _round_trip * _cost_scale
if _cost_aware:
    _y_raw, _ = build_cost_aware_forward_labels(
        close, _PRIMARY_FB, round_trip_cost=_effective_cost
    )
else:
    _y_raw = build_forward_labels(close, _PRIMARY_FB)

_work = df_feat[list(feat_cols)].copy()
_work["target"] = _y_raw.values
_work = _work.replace([np.inf, -np.inf], np.nan).dropna()
if len(_work) < 500 and _cost_aware:
    _y_gross = build_forward_labels(close, _PRIMARY_FB)
    _work = df_feat[list(feat_cols)].copy()
    _work["target"] = _y_gross.values
    _work = _work.replace([np.inf, -np.inf], np.nan).dropna()

_train_idx, _embargo_idx, _val_idx, _ = chronological_split_indices(
    len(_work),
    validation_fraction=_val_frac,
    embargo_bars=_PRIMARY_FB,
)
X_val = _work.iloc[_val_idx][list(feat_cols)]
y_val = _work.iloc[_val_idx]["target"].values.astype(np.float64)

_primary_head = multi_bundle.get_head(_PRIMARY_FB)
_inference = _primary_vm.get("inference_path", "regressor_mean")
if _inference == "regressor_mean":
    val_preds = np.asarray(
        _primary_head.predict(
            X_val.values,
            X_df=X_val,
            inference_stack="regressor_mean",
        ),
        dtype=np.float64,
    ).ravel()
else:
    val_preds = np.asarray(
        _primary_head.predict(X_val.values, X_df=X_val),
        dtype=np.float64,
    ).ravel()

_short_cut = -abs(_short_thr)
position = np.zeros(len(val_preds), dtype=np.int8)
position[val_preds > _long_thr] = 1
position[val_preds < _short_cut] = -1

gross_ret = np.where(position == 1, y_val, np.where(position == -1, -y_val, 0.0))
# Charge round-trip cost once per position entry (not every bar held).
_pos_changes = np.diff(np.concatenate([[0], position.astype(np.int16)]))
_entries = _pos_changes != 0
_entry_cost = np.where(_entries & (position != 0), _round_trip, 0.0)
net_ret = np.where(position != 0, gross_ret - _entry_cost, 0.0)
active = position != 0

cum_pnl = np.cumsum(net_ret)
peak = np.maximum.accumulate(cum_pnl)
max_dd = float(np.min(cum_pnl - peak)) if len(cum_pnl) else 0.0
total_return = float(cum_pnl[-1]) if len(cum_pnl) else 0.0
win_rate = float(np.mean(net_ret[active] > 0)) if active.any() else 0.0
n_trades = int(active.sum())

_candle_minutes = 5  # base 5m candle grid
_bars_per_year = 365.0 * (1440.0 / _candle_minutes) / max(1, _PRIMARY_FB)
_active_net = net_ret[active]
if _active_net.size >= 2 and np.std(_active_net) > 1e-12:
    sharpe = float(np.mean(_active_net) / np.std(_active_net) * np.sqrt(_bars_per_year))
else:
    sharpe = 0.0

print("\n" + "=" * 72)
print(f"Simulated validation P&L — {_PRIMARY_KEY} ({_PRIMARY_FB} bars, n={len(val_preds)})")
print("=" * 72)
print(f"  long_threshold (p90):     {_long_thr:.6f}")
print(f"  short_threshold (|p10|): {_short_thr:.6f}  (signal if pred < {-_short_thr:.6f})")
print(f"  active bars (trades):     {n_trades} / {len(val_preds)} ({100.0 * n_trades / max(1, len(val_preds)):.1f}% coverage)")
print(f"  cumulative_return:        {total_return * 100:.3f}%")
print(f"  win_rate (net, active):   {win_rate * 100:.2f}%")
print(f"  Sharpe (annualized est.): {sharpe:.2f}")
print(f"  max_drawdown:             {max_dd * 100:.3f}%")
print(f"  validation_corr:          {_primary_vm.get('validation_corr')}")

# --- 3) Matplotlib 2x2 dashboard (optional) ---
try:
    import matplotlib.pyplot as plt

    _corr_vals = []
    _corr_labels = []
    for _hkey in V43_HORIZON_KEYS:
        _vm_h = metadata["horizons"][_hkey]["validation_metrics"]
        _c = _vm_h.get("validation_corr")
        if _c is not None:
            _corr_vals.append(float(_c))
            _corr_labels.append(_hkey.replace("_", "\n"))

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("JackSparrow v43 — pre-export performance", fontsize=13)

    ax0 = axes[0, 0]
    _colors = ["#2ca02c" if c >= 0 else "#d62728" for c in _corr_vals]
    ax0.bar(_corr_labels, _corr_vals, color=_colors)
    ax0.axhline(0.0, color="black", linewidth=0.8)
    ax0.set_title("Validation correlation by horizon")
    ax0.set_ylabel("corr")
    ax0.tick_params(axis="x", labelsize=8)

    ax1 = axes[0, 1]
    ax1.plot(cum_pnl * 100.0, color="#1f77b4", linewidth=1.2)
    ax1.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax1.set_title(f"Cumulative P&L — {_PRIMARY_KEY}")
    ax1.set_xlabel("validation bar index")
    ax1.set_ylabel("cumulative return (%)")
    ax1.text(
        0.02,
        0.95,
        f"Sharpe~{sharpe:.2f}  MaxDD={max_dd * 100:.2f}%",
        transform=ax1.transAxes,
        va="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )

    ax2 = axes[1, 0]
    _x = np.arange(len(V43_HORIZON_KEYS))
    _w = 0.35
    _l_hits = []
    _s_hits = []
    for _hkey in V43_HORIZON_KEYS:
        _vm_h = metadata["horizons"][_hkey]["validation_metrics"]
        _l_hits.append(float((_vm_h.get("long_candidates") or {}).get("hit_rate_net_positive") or 0.5))
        _s_hits.append(float((_vm_h.get("short_candidates") or {}).get("hit_rate_net_positive") or 0.5))
    ax2.bar(_x - _w / 2, _l_hits, _w, label="long hit (net)", color="#2ca02c")
    ax2.bar(_x + _w / 2, _s_hits, _w, label="short hit (net)", color="#ff7f0e")
    ax2.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="50% baseline")
    ax2.set_xticks(_x)
    ax2.set_xticklabels([k.replace("_", "\n") for k in V43_HORIZON_KEYS], fontsize=8)
    ax2.set_ylim(0.0, 1.0)
    ax2.set_title("Candidate hit rates (validation)")
    ax2.legend(fontsize=8)

    ax3 = axes[1, 1]
    ax3.hist(val_preds, bins=40, color="#9467bd", alpha=0.75, edgecolor="white")
    ax3.axvline(_long_thr, color="#2ca02c", linestyle="--", linewidth=1.2, label="long thr")
    ax3.axvline(-_short_thr, color="#d62728", linestyle="--", linewidth=1.2, label="short thr")
    ax3.set_title(f"Prediction distribution — {_PRIMARY_KEY}")
    ax3.set_xlabel("predicted forward return")
    ax3.legend(fontsize=8)

    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n(matplotlib not installed — skipping charts; metrics above are complete.)")

## 5) `FeatureEngineer` + export

`FeatureEngineer` from `v43_pickle_shims` stores only the canonical column contract. The actual transform path is registered with `set_v43_build_feature_matrix(...)`, matching the agent startup path and avoiding fragile notebook-local closure pickling.


## Export gates (production vs research)

- **Production (Colab default)**: `V43_EXPORT_GATES_STRICT=true` + **primary-only** — gates use `validation_corr` and candidate stats (no `meta_auc` requirement for `regressor_mean` exports).
- **Full strict (all four heads)**: `os.environ["V43_EXPORT_STRICT_PRIMARY_ONLY"] = "false"`.
- **Research relaxed**: `V43_EXPORT_GATES_STRICT=false`.
- **Runtime**: set `JACKSPARROW_V43_INFERENCE_STACK=regressor_mean` on the agent to match this notebook export.
- **Do not promote** bundles to `agent/model_storage/` unless strict gates pass without env overrides.

Default history: **200,000** × 5m bars (~694 days) + **16,000** funding hours. Override with `TARGET_CANDLES_5M` / `FUNDING_LOOKBACK_HOURS` if needed.


In [ ]:
# OI / microstructure data quality (diagnostic — does not block export)
from feature_store.jacksparrow_v43_oi_history import oi_feature_matrix_diagnostics

_OI_FEAT_COLS = (
    "oi_zscore",
    "oi_change_6",
    "oi_price_divergence",
    "oi_acceleration",
    "oi_delta_z",
)
_MICRO_COLS = (
    "bid_ask_imbalance",
    "spread_bps",
    "funding_x_oi",
    "funding_predicted_zscore",
)
_oi_std = oi_feature_matrix_diagnostics(df_feat)
print("OI feature std at export:", _oi_std)
if max(_oi_std.values()) < 1e-8:
    print(
        "WARNING: OI model features have no variance — bundle lacks OI signal. "
        "Re-run with V43_FETCH_HISTORICAL_OI=true and aligned df_5m."
    )
_micro_present = [c for c in _MICRO_COLS if c in df_feat.columns]
if _micro_present:
    _micro_zero = {
        c: float((pd.to_numeric(df_feat[c], errors="coerce").fillna(0.0).abs() < 1e-12).mean())
        for c in _micro_present
    }
    _mz = max(_micro_zero.values())
    print(f"Microstructure zero fraction (expected high without ticker CSV): max={_mz * 100:.1f}%")
    if _mz >= 0.80:
        print(
            "NOTE: bid/ask microstructure zero-filled — optional V43_TICKER_SNAPSHOTS_CSV for production."
        )


def make_feature_engineer():
    fe = FeatureEngineer()
    fe.columns = list(V43_CANONICAL_FEATURES)
    return fe


def _version_or_none(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def _git_commit_or_none():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=str(REPO_ROOT),
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
    except Exception:
        return None


set_v43_build_feature_matrix(build_v43_feature_matrix)
fe = make_feature_engineer()

OUT_DIR = Path(os.environ.get("V43_EXPORT_DIR", "jacksparrow_v43_bundle"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
artifact_path = OUT_DIR / "model_artifact_v43.pkl"
metadata_path = OUT_DIR / "metadata_v43.json"

assert validation_metrics.get("inference_path") in (
    "regressor_mean",
    "meta_calibrator",
), (
    "train_multihead_from_feature_matrix must set inference_path to regressor_mean "
    "(default) or meta_calibrator when V43_USE_META_STACK=true."
)
if validation_metrics.get("inference_path") == "regressor_mean":
    print("Export stack: regressor_mean (no meta-classifier in artifact)")
from feature_store.jacksparrow_v43_multihead import (
    format_export_gate_summary,
    resolve_min_meta_auc_by_horizon,
    validate_multihead_export_gates,
)

# Export gates: production defaults live in the shared library.
# Colab runs treat ALL export gates as non-blocking diagnostics — they never raise.
# For stricter research runs, set env vars like V43_MIN_META_AUC_SCALP_10M /
# V43_MIN_VALIDATION_CORR_SCALP_10M before re-running this cell.
_export_strict = os.environ.get("V43_EXPORT_GATES_STRICT", "true").strip().lower() in (
    "1", "true", "yes",
)
print(format_export_gate_summary(metadata, strict_primary_only=True))
print(f"V43_EXPORT_GATES_STRICT={_export_strict}")
print("V43_EXPORT_STRICT_PRIMARY_ONLY=True (diagnostics only in this notebook)")
_scalp_inf = metadata["horizons"]["scalp_10m"]["validation_metrics"].get(
    "inference_path", "regressor_mean"
)
print(
    f"Primary stack ({_scalp_inf}): gates use validation_corr; "
    "meta_auc enforced when inference_path=meta_calibrator."
)
gate_failures, gate_warnings = validate_multihead_export_gates(
    metadata,
    strict=False,  # Colab: never hard-block on gates; treat everything as diagnostics
    return_soft=True,
    strict_primary_only=True,
)
for _gw in gate_warnings:
    print(f"EXPORT WARN: {_gw}")
if gate_failures:
    for _gf in gate_failures:
        print(f"EXPORT GATE (diagnostic only): {_gf}")
if gate_warnings and not _export_strict:
    print("Export allowed with warnings (notebook-only; use stricter gates in production tooling).")

primary_vm = metadata["horizons"]["scalp_10m"]["validation_metrics"]
if primary_vm.get("meta_auc") is not None:
    _mins = resolve_min_meta_auc_by_horizon()
    print(
        f"Primary horizon scalp_10m: meta_auc={float(primary_vm.get('meta_auc')):.4f} "
        f"(min {_mins['scalp_10m']:.2f}), "
        f"corr={float(primary_vm.get('validation_corr')):.4f}"
    )
else:
    print(
        f"Primary horizon scalp_10m: corr={float(primary_vm.get('validation_corr')):.4f} "
        f"(regressor_mean — no meta_auc)"
    )

_long_hr = primary_vm["long_candidates"].get("hit_rate_gross_positive") or 0.0
_short_hr = primary_vm["short_candidates"].get("hit_rate_gross_positive") or 0.0
_hr_failures = []
if _long_hr < 0.50:
    _hr_failures.append(f"long_hit_rate_gross={_long_hr:.4f} < 0.50")
if _short_hr < 0.50:
    _hr_failures.append(f"short_hit_rate_gross={_short_hr:.4f} < 0.50")
if _hr_failures:
    print("HIT RATE WARN (diagnostic only): " + "; ".join(_hr_failures))

artifact_payload = artifact_dict_from_bundle(multi_bundle, fe)
artifact_payload.update({
    "feature_engineer": fe,
    "features": list(V43_CANONICAL_FEATURES),
    })

joblib.dump(artifact_payload, artifact_path)

_agent_load_snippet = "\n".join(
    [
        "artifact = joblib.load(MODEL_ARTIFACT_PATH)",
        "model    = artifact[\"model\"]",
        "features = artifact[\"features\"]",
        "fe       = artifact[\"feature_engineer\"]",
        "df_feat  = fe.transform(df5, df15, df1h, df_funding, include_target=False)",
        "assert len(df_feat) >= 2, \"Need at least 2 rows to use the last closed bar\"",
        "X        = df_feat[features].iloc[[-2]].values",
        "X_df     = df_feat[features].iloc[[-2]]",
        "expected_return = model.predict_head(2, X, X_df=X_df)  # primary scalp_10m head",
    ]
)

meta = dict(metadata)
meta.update({
    "version": "v43",
    "compatible_feature_version": V43_COMPATIBLE_FEATURE_VERSION,
    "symbol": SYMBOL,
    "model_name": "jacksparrow_v43_" + str(SYMBOL),
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "feature_count": len(V43_CANONICAL_FEATURES),
    "features": list(V43_CANONICAL_FEATURES),
    "model_family": V43_MULTIHEAD_MODEL_FAMILY,
    "artifact_format": V43_MULTIHEAD_ARTIFACT_FORMAT,
    "primary_execution_horizon_bars": 2,
    "target_definition": metadata.get("target_definition", "cost_aware_forward_return"),
    "model_architecture": {
        "base_estimators": ["lgbm_regressor", "xgb_regressor", "rf_regressor"],
        "inference_stack": primary_vm.get("inference_path", "regressor_mean"),
        "meta_learner": None,
        "calibrator": None,
    },
    "split": split_metadata,
    "data": {
        "delta_base_url": DELTA_BASE,
        "symbol": SYMBOL,
        "resolution": RESOLUTION_5M,
        "target_candles_5m": TARGET_CANDLES_5M,
        "funding_lookback_hours": FUNDING_LOOKBACK_HOURS,
        "candles_5m_rows": int(len(df_5m)),
        "funding_rows": int(len(df_funding)),
        "candles_5m_start": df_5m["timestamp"].min().isoformat() if len(df_5m) else None,
        "candles_5m_end": df_5m["timestamp"].max().isoformat() if len(df_5m) else None,
    },
    "provenance": {
        "repo_root": str(REPO_ROOT),
        "git_commit": _git_commit_or_none(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": {
            "joblib": _version_or_none("joblib"),
            "numpy": _version_or_none("numpy"),
            "pandas": _version_or_none("pandas"),
            "scikit-learn": _version_or_none("scikit-learn"),
            "lightgbm": _version_or_none("lightgbm"),
            "xgboost": _version_or_none("xgboost"),
        },
    },
    "agent_load": _agent_load_snippet,
})
metadata_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Wrote", artifact_path)
print("Wrote", metadata_path)



## 6) Alignment self-check + handoff

Run `validate_v43_metadata_compatibility` on the exported JSON. Then copy the bundle folder into
`agent/model_storage/` (for example `JackSparrow_v43_models_BTCUSD/`) and configure model discovery per
`docs/03-ml-models.md`.

**On your machine (after download):**

```text
pytest tests/unit/test_jacksparrow_v43_contract.py
python scripts/smoke_test_v43.py
pytest tests/unit/test_jack_sparrow_v43_node_ctx.py
```


In [ ]:
with metadata_path.open(encoding="utf-8") as f:
    meta_loaded = json.load(f)
validate_v43_metadata_compatibility(meta_loaded)

set_v43_build_feature_matrix(build_v43_feature_matrix)
tail5 = df_5m.tail(800).copy()
sm = fe.transform(tail5, None, None, df_funding, df_oi=df_oi, df_mark=df_mark, include_target=False)
assert len(sm) >= 2
assert list(sm.columns[: len(V43_CANONICAL_FEATURES)]) == list(V43_CANONICAL_FEATURES)
_feat_list = list(V43_CANONICAL_FEATURES)
expected_return = multi_bundle.get_head(2).predict(
    sm[_feat_list].iloc[[-2]].values,
    X_df=sm[_feat_list].iloc[[-2]],
)
assert np.atleast_1d(np.asarray(expected_return, dtype=np.float64)).size == 1
assert "horizons" in meta_loaded and "intraday_30m" in meta_loaded["horizons"]
assert "validation_metrics" in meta_loaded["horizons"]["intraday_30m"]
assert "split" in meta_loaded
print("validate_v43_metadata_compatibility: OK")
print("metadata split + validation metrics: OK")
print("fe.transform + multi_bundle.get_head(2).predict smoke: OK", expected_return)



In [ ]:
try:
    from google.colab import files
    import shutil

    z = Path("/content/jacksparrow_v43_bundle.zip")
    shutil.make_archive(str(z.with_suffix("")), "zip", root_dir=str(OUT_DIR))
    files.download(str(z))
except Exception as e:
    print("Colab zip/download skipped:", e)

